In [1]:
from pathlib import Path

nnunet_dir = Path("/nfs/data/nii/data1/Analysis/zanderch___HU_Messung/ANALYSIS_seg/nnunet")
dataset_name = "Dataset001_Vertebrae"

imgs_dir     = nnunet_dir / "nnUNet_raw" / dataset_name / "imagesTr"
roi_dir      = nnunet_dir / "nnUNet_raw" / dataset_name / "labelsTr"
vertebrae_dir = nnunet_dir / "nnUNet_raw" / dataset_name / "l1_segmentations"

In [ ]:
import random
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

N_SAMPLES = 4
HALF = 32  # half-side of the 64³ crop

cases = sorted(
    p.name for p in vertebrae_dir.iterdir()
    if (vertebrae_dir / p.name / "vertebrae_L1.nii.gz").exists()
    and (roi_dir      / f"{p.name}.nii.gz").exists()
    and (imgs_dir     / f"{p.name}_0000.nii.gz").exists()
)
print(f"{len(cases)} complete cases found")
sample_cases = random.sample(cases, min(N_SAMPLES, len(cases)))


def crop_slice(c, maxdim):
    lo = max(0, min(c - HALF, maxdim - 2 * HALF))
    return slice(lo, lo + 2 * HALF)


fig, axes = plt.subplots(len(sample_cases), 3, figsize=(14, 5 * len(sample_cases)))
axes = np.atleast_2d(axes)

for row, case in enumerate(sample_cases):
    img_data = nib.load(imgs_dir      / f"{case}_0000.nii.gz").get_fdata()
    roi_data = nib.load(roi_dir       / f"{case}.nii.gz").get_fdata().astype(bool)
    l1_data  = nib.load(vertebrae_dir / case / "vertebrae_L1.nii.gz").get_fdata().astype(bool)

    coords = np.argwhere(l1_data)
    cx, cy, cz = (coords.mean(axis=0).astype(int) if len(coords) else
                  np.array(img_data.shape) // 2)

    sx = crop_slice(cx, img_data.shape[0])
    sy = crop_slice(cy, img_data.shape[1])
    sz = crop_slice(cz, img_data.shape[2])

    img_c = img_data[sx, sy, sz]
    roi_c = roi_data[sx, sy, sz]
    l1_c  = l1_data[ sx, sy, sz]

    mx, my, mz = HALF, HALF, HALF

    views = [
        ("Axial",    img_c[:, :, mz].T, roi_c[:, :, mz].T, l1_c[:, :, mz].T),
        ("Coronal",  img_c[:, my, :].T, roi_c[:, my, :].T, l1_c[:, my, :].T),
        ("Sagittal", img_c[mx, :, :].T, roi_c[mx, :, :].T, l1_c[mx, :, :].T),
    ]

    for col, (label, img_sl, roi_sl, l1_sl) in enumerate(views):
        ax = axes[row, col]
        ax.imshow(img_sl, cmap="gray", vmin=-200, vmax=800, origin="lower")
        ax.imshow(np.ma.masked_where(~roi_sl, roi_sl), cmap="Reds",  alpha=0.45, vmin=0, vmax=1, origin="lower")
        ax.imshow(np.ma.masked_where(~l1_sl,  l1_sl),  cmap="Blues", alpha=0.60, vmin=0, vmax=1, origin="lower")
        ax.set_title(f"{case[:18]}\n{label}" if col == 0 else label, fontsize=8)
        ax.axis("off")

fig.legend(
    handles=[Patch(color="salmon",    alpha=0.6, label="ROI"),
             Patch(color="steelblue", alpha=0.7, label="vertebrae_L1")],
    loc="lower center", ncol=2, fontsize=10, frameon=False,
)
plt.tight_layout(rect=[0, 0.04, 1, 1])
plt.show()